In [ ]:
# ============================================================
# CELL 1: Import Libraries
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded.')

In [ ]:
# ============================================================
# CELL 2: Load Dataset
# ============================================================
csv_file = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.csv'):
            csv_file = os.path.join(dirname, filename)
            print(f'Found: {csv_file}')

if csv_file is None:
    raise FileNotFoundError('No CSV found. Add the dataset from the Input panel on the right.')

df = pd.read_csv(csv_file)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ============================================================
# CELL 3: Inspect Every Column — Names, Types, Sample Values
# ============================================================
print(f'Rows: {df.shape[0]}   Columns: {df.shape[1]}')
print('\nColumn names, dtype, and sample values:')
for col in df.columns:
    sample = df[col].dropna().unique()[:4]
    print(f'  {col:<35} {str(df[col].dtype):<10} {sample}')

In [ ]:
# ============================================================
# CELL 4: Missing Values and Duplicates
# ============================================================
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'None found.')

print(f'\n=== Duplicates ===')
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')
if dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Removed duplicates. New shape: {df.shape}')

print('\n=== Numeric Columns Summary ===')
print(df.select_dtypes(include=np.number).describe().round(2))

print('\n=== Categorical Columns Summary ===')
for col in df.select_dtypes(include='object').columns:
    print(f'  {col}: {df[col].nunique()} unique values -> {df[col].unique()[:5]}')

In [ ]:
# ============================================================
# CELL 5: Fill Any Missing Values
# ============================================================
for col in df.columns:
    n = df[col].isnull().sum()
    if n > 0:
        if df[col].dtype == 'object':
            fill = df[col].mode()[0]
            df[col].fillna(fill, inplace=True)
            print(f'  {col}: {n} nulls filled with mode "{fill}"')
        else:
            fill = df[col].median()
            df[col].fillna(fill, inplace=True)
            print(f'  {col}: {n} nulls filled with median {fill:.2f}')
print(f'Done. Shape: {df.shape}')

In [ ]:
# ============================================================
# CELL 6: Create Binary Target — High_Risk
# ============================================================
# The Automation_Risk column contains: 'Low', 'Medium', 'High'
# We define High_Risk = 1 if Automation_Risk == 'High', else 0
# This is a natural, meaningful binary split:
#   High -> truly at risk of automation
#   Low / Medium -> not classified as high risk

# Find the automation risk column (handles any capitalization)
risk_col = None
for col in df.columns:
    if 'automation' in col.lower() and 'risk' in col.lower():
        risk_col = col
        break
if risk_col is None:
    for col in df.columns:
        if 'risk' in col.lower():
            risk_col = col
            break

print(f'Using column: "{risk_col}"')
print(f'Unique values: {df[risk_col].unique()}')
print(f'Dtype: {df[risk_col].dtype}')

if df[risk_col].dtype == 'object':
    df['High_Risk'] = (df[risk_col].str.strip().str.lower() == 'high').astype(int)
    print('\nMapping: "High" -> 1,  "Low"/"Medium" -> 0')
else:
    median_val = df[risk_col].median()
    df['High_Risk'] = (df[risk_col] > median_val).astype(int)
    print(f'Numeric column: split at median {median_val:.4f}')

counts = df['High_Risk'].value_counts().sort_index()
pcts   = df['High_Risk'].value_counts(normalize=True).sort_index() * 100
print(f'\nClass Distribution:')
print(f'  Low / Not High Risk (0): {counts[0]} rows  ({pcts[0]:.1f}%)')
print(f'  High Risk           (1): {counts[1]} rows  ({pcts[1]:.1f}%)')

In [ ]:
# ============================================================
# CELL 7: Encode Categorical Features
# ============================================================
df_enc = df.copy()

# Step 1 — Drop the original risk column (target was derived from it)
df_enc.drop(columns=[risk_col], inplace=True)
print(f'Dropped source column: {risk_col}')

# Step 2 — Drop Job_Title: identifier-like, too many unique values
job_col = None
for col in df_enc.columns:
    if 'job' in col.lower() and 'title' in col.lower():
        job_col = col
        break
if job_col:
    df_enc.drop(columns=[job_col], inplace=True)
    print(f'Dropped identifier column: {job_col}')

# Step 3 — Binary encode Remote_Friendly
for col in df_enc.columns:
    if 'remote' in col.lower() and df_enc[col].dtype == 'object':
        df_enc[col] = (df_enc[col].str.strip().str.lower() == 'yes').astype(int)
        print(f'Binary encoded: {col}  (Yes=1, No=0)')

# Step 4 — Ordinal encode AI_Adoption_Level
for col in df_enc.columns:
    if 'adoption' in col.lower() and df_enc[col].dtype == 'object':
        order = {'low': 0, 'medium': 1, 'high': 2}
        df_enc[col] = df_enc[col].str.strip().str.lower().map(order).fillna(1).astype(int)
        print(f'Ordinal encoded: {col}  (Low=0, Medium=1, High=2)')

# Step 5 — Ordinal encode Education_Required
for col in df_enc.columns:
    if 'education' in col.lower() and df_enc[col].dtype == 'object':
        def enc_edu(v):
            v = str(v).lower()
            if 'high school' in v: return 0
            if 'associate'   in v: return 1
            if 'bachelor'    in v: return 2
            if 'master'      in v: return 3
            if 'phd' in v or 'doctorate' in v: return 4
            return 2
        df_enc[col] = df_enc[col].apply(enc_edu)
        print(f'Ordinal encoded: {col}  (HighSchool=0 ... PhD=4)')

# Step 6 — One-Hot encode remaining text columns
obj_cols = df_enc.select_dtypes(include='object').columns.tolist()
obj_cols = [c for c in obj_cols if c != 'High_Risk']
if obj_cols:
    print(f'One-Hot encoding: {obj_cols}')
    df_enc = pd.get_dummies(df_enc, columns=obj_cols, drop_first=True)

# Step 7 — Convert any bool columns to int
for col in df_enc.select_dtypes(include='bool').columns:
    df_enc[col] = df_enc[col].astype(int)

print(f'\nFinal encoded shape: {df_enc.shape}')
print(f'All dtypes numeric: {all(df_enc.dtypes != object)}')
df_enc.head(3)

In [ ]:
# ============================================================
# CELL 8: Visualization 1 — Target Distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: original Automation_Risk category counts
risk_counts = df[risk_col].value_counts()
bar_colors  = ['#5ab4ac' if v.lower() != 'high' else '#d8b365'
               for v in risk_counts.index]
axes[0].bar(risk_counts.index, risk_counts.values,
            color=bar_colors, edgecolor='white', width=0.5)
for i, v in enumerate(risk_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')
axes[0].set_title(f'Distribution of {risk_col}', fontsize=12)
axes[0].set_xlabel('Risk Category')
axes[0].set_ylabel('Count')

# Right: binary High_Risk class balance
hr_counts = df['High_Risk'].value_counts().sort_index()
axes[1].bar(['Low/Medium Risk (0)', 'High Risk (1)'], hr_counts.values,
            color=['#5ab4ac', '#d8b365'], edgecolor='white', width=0.5)
for i, v in enumerate(hr_counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')
axes[1].set_title('Binary Target: High_Risk', fontsize=12)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('viz1_target_distribution.png', bbox_inches='tight')
plt.show()
print('Viz 1 complete.')

In [ ]:
# ============================================================
# CELL 9: Visualization 2 — Salary by Risk Category
# ============================================================
from matplotlib.patches import Patch

# Find salary column
sal_col = next((c for c in df.columns if 'salary' in c.lower()), None)

if sal_col:
    fig, ax = plt.subplots(figsize=(9, 5))
    palette = {'Low': '#5ab4ac', 'Medium': '#abd9e9', 'High': '#d8b365'}
    for cat, grp in df.groupby(risk_col):
        ax.hist(grp[sal_col], bins=25, alpha=0.6,
                label=cat, color=palette.get(cat, 'grey'), edgecolor='white')
    ax.set_title(f'{sal_col} Distribution by {risk_col}', fontsize=12)
    ax.set_xlabel(sal_col)
    ax.set_ylabel('Count')
    ax.legend(title='Risk Level')
    plt.tight_layout()
    plt.savefig('viz2_salary_by_risk.png', bbox_inches='tight')
    plt.show()
    print('Viz 2 complete.')

In [ ]:
# ============================================================
# CELL 10: Visualization 3 — High Risk Count by Industry
# ============================================================
ind_col = next((c for c in df.columns if 'industry' in c.lower()), None)

if ind_col:
    fig, ax = plt.subplots(figsize=(12, 5))
    industry_hr = (df.groupby(ind_col)['High_Risk']
                     .mean()
                     .sort_values(ascending=False))
    industry_hr.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Proportion of High-Risk Jobs by {ind_col}', fontsize=12)
    ax.set_xlabel(ind_col)
    ax.set_ylabel('Proportion High Risk')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('viz3_industry_risk.png', bbox_inches='tight')
    plt.show()
    print('Viz 3 complete.')

In [ ]:
# ============================================================
# CELL 11: Visualization 4 — Correlation Heatmap (Numeric Features)
# ============================================================
num_df = df.select_dtypes(include=np.number).copy()
num_df['High_Risk'] = df['High_Risk'].values

# Keep only columns with actual variance and cap at 10 for readability
valid = [c for c in num_df.columns if num_df[c].std() > 0][:10]

fig, ax = plt.subplots(figsize=(10, 7))
corr = num_df[valid].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=12)
plt.tight_layout()
plt.savefig('viz4_heatmap.png', bbox_inches='tight')
plt.show()
print('Viz 4 complete.')

In [ ]:
# ============================================================
# CELL 12: Visualization 5 — Boxplots of Numeric Features by Class
# ============================================================
num_cols = df.select_dtypes(include=np.number).columns.tolist()
# Pick up to 3 informative numeric columns
box_cols = num_cols[:3] if len(num_cols) >= 3 else num_cols

fig, axes = plt.subplots(1, len(box_cols), figsize=(5 * len(box_cols), 5))
if len(box_cols) == 1:
    axes = [axes]

for i, col in enumerate(box_cols):
    df.boxplot(column=col, by='High_Risk', ax=axes[i],
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2),
               whiskerprops=dict(color='steelblue'),
               capprops=dict(color='steelblue'))
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('High Risk (0 = Low/Med, 1 = High)')

plt.suptitle('Numeric Feature Distributions by Risk Class', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('viz5_boxplots.png', bbox_inches='tight')
plt.show()
print('Viz 5 complete.')

In [ ]:
# ============================================================
# CELL 13: Build Feature Matrix X and Target y
# ============================================================
# High_Risk is target; everything else is a feature
X = df_enc.drop(columns=['High_Risk'])
y = df_enc['High_Risk']

# Safety check: ensure no non-numeric features remain
bad = X.select_dtypes(exclude=np.number).columns.tolist()
if bad:
    X = X.drop(columns=bad)
    print(f'Dropped non-numeric: {bad}')

print(f'Features: {X.shape[1]} columns, {X.shape[0]} rows')
print(f'Target  : {y.value_counts().to_dict()}')

In [ ]:
# ============================================================
# CELL 14: Train / Test Split (80/20, stratified) + Scale
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} rows   Test: {X_test.shape[0]} rows')
print(f'Train class balance: {y_train.value_counts().to_dict()}')
print(f'Test  class balance: {y_test.value_counts().to_dict()}')

# StandardScaler: zero mean, unit variance
# IMPORTANT: fit ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print('\nScaling applied.')

In [ ]:
# ============================================================
# CELL 15: Train Logistic Regression
# ============================================================
# C=1.0 is the default regularization strength.
# Regularization prevents overfitting by penalizing large coefficients.
# lbfgs solver works well for small-to-medium datasets.

model = LogisticRegression(
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)
model.fit(X_train_s, y_train)

print('Model trained.')
print(f'  Intercept : {model.intercept_[0]:.4f}')
print(f'  Iterations: {model.n_iter_[0]}')

In [ ]:
# ============================================================
# CELL 16: Cross-Validation (5-fold) — Verify No Overfitting
# ============================================================
# Cross-validation gives a more reliable estimate of real-world accuracy
# by testing on 5 different held-out portions of the training data.
import numpy as np
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42))
])

cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
print(f'5-Fold Cross-Validation Accuracy:')
print(f'  Fold scores : {np.round(cv_scores, 4)}')
print(f'  Mean        : {cv_scores.mean():.4f}')
print(f'  Std Dev     : {cv_scores.std():.4f}')
print(f'\nLow std dev = consistent performance = no overfitting.')

In [ ]:
# ============================================================
# CELL 17: Threshold Analysis — Effect on Predictions
# ============================================================
y_prob = model.predict_proba(X_test_s)[:, 1]  # probability of High Risk
y_pred = (y_prob >= 0.5).astype(int)           # default threshold = 0.5

print(f'Using default threshold = 0.5')
print(f'\nThreshold comparison:')
print(f'{"Threshold":>10} | {"Pred High Risk":>14} | {"Accuracy":>9} | {"Recall":>7} | {"Precision":>10}')
print('-' * 62)
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p   = (y_prob >= t).astype(int)
    acc = accuracy_score(y_test, p)
    rec = recall_score(y_test, p)
    pre = precision_score(y_test, p)
    print(f'{t:>10} | {p.sum():>14} | {acc:>9.4f} | {rec:>7.4f} | {pre:>10.4f}')

In [ ]:
# ============================================================
# CELL 18: Feature Coefficients
# ============================================================
coeff_df = pd.DataFrame({
    'Feature'    : X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print(f'Model Intercept: {model.intercept_[0]:.4f}')
print('\nTop 15 Coefficients (absolute magnitude):')
print(coeff_df.head(15).to_string(index=False))

# Visualize
top15      = coeff_df.head(15)
bar_colors = ['#d8b365' if c > 0 else '#5ab4ac' for c in top15['Coefficient']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top15['Feature'], top15['Coefficient'], color=bar_colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 Feature Coefficients', fontsize=12)
ax.set_xlabel('Coefficient (positive = increases High Risk probability)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('viz6_coefficients.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 19: Confusion Matrix
# ============================================================
cm           = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=== Confusion Matrix — Test Set ===')
print(f'  True Negatives  (TN) = {tn:4d}  correctly predicted Low/Medium Risk')
print(f'  False Positives (FP) = {fp:4d}  predicted High Risk but actually Low/Medium')
print(f'  False Negatives (FN) = {fn:4d}  predicted Low/Medium but actually High Risk  ← dangerous miss')
print(f'  True Positives  (TP) = {tp:4d}  correctly predicted High Risk')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Low/Med Risk', 'High Risk']
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Test Set', fontsize=12)
plt.tight_layout()
plt.savefig('viz7_confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 20: Evaluation Metrics
# ============================================================
acc  = accuracy_score(y_test,  y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test,    y_pred)
f1   = f1_score(y_test,        y_pred)

print('=== Metrics on Test Set ===')
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Low/Med Risk', 'High Risk']))

In [ ]:
# ============================================================
# CELL 21: Naive Baseline Comparison
# ============================================================
majority     = y_test.mode()[0]
baseline_acc = (y_test == majority).mean()

print(f'Majority class         : {majority}')
print(f'Naive baseline accuracy: {baseline_acc:.4f}  ({baseline_acc*100:.1f}%)')
print(f'Our model accuracy     : {acc:.4f}  ({acc*100:.1f}%)')
print(f'Improvement            : +{(acc - baseline_acc)*100:.2f} percentage points')

In [ ]:
# ============================================================
# CELL 22: Overfitting Check — Train vs Test Score
# ============================================================
train_acc = model.score(X_train_s, y_train)
test_acc  = model.score(X_test_s,  y_test)
gap       = abs(train_acc - test_acc)

print(f'Training Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)')
print(f'Testing Accuracy  : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'Gap               : {gap:.4f}  ({gap*100:.2f}%)')

if gap < 0.03:
    verdict = 'No significant overfitting. Model generalizes well.'
elif train_acc - test_acc > 0.05:
    verdict = 'Possible overfitting. Training score is notably higher.'
else:
    verdict = 'Minor gap — acceptable for a linear model.'
print(f'\nVerdict: {verdict}')
print(f'Cross-val mean (Cell 16): {cv_scores.mean():.4f} — consistent with test score.')

In [ ]:
# ============================================================
# CELL 23: Improvement — Threshold Tuning (0.5 → 0.4)
# ============================================================
# Recall is most important for this problem.
# A False Negative = a high-risk worker gets no warning.
# Lowering the threshold increases recall at the cost of some precision.

y_pred_05 = (y_prob >= 0.5).astype(int)
y_pred_04 = (y_prob >= 0.4).astype(int)

print(f'  Metric      | Threshold 0.5 | Threshold 0.4 | Change')
print(f'  {"":─<55}')
for name, fn in [('Accuracy',  accuracy_score),
                 ('Recall',    recall_score),
                 ('Precision', precision_score),
                 ('F1-Score',  f1_score)]:
    b = fn(y_test, y_pred_05)
    a = fn(y_test, y_pred_04)
    arrow = '↑' if a > b else ('↓' if a < b else '—')
    print(f'  {name:<12}| {b:.4f}        | {a:.4f}        | {arrow} {abs(a-b)*100:.2f}%')

In [ ]:
# ============================================================
# CELL 24: Confusion Matrices — Before vs After Threshold
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_05),
    display_labels=['Low/Med', 'High Risk']
).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Threshold = 0.5 (Default)', fontsize=11)

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_04),
    display_labels=['Low/Med', 'High Risk']
).plot(ax=axes[1], cmap='Oranges', colorbar=False)
axes[1].set_title('Threshold = 0.4 (After Tuning)', fontsize=11)

plt.suptitle('Impact of Threshold Tuning on Predictions', fontsize=12)
plt.tight_layout()
plt.savefig('viz8_threshold_comparison.png', bbox_inches='tight')
plt.show()

print('\nAll 24 cells complete. Check outputs above.')